# Notebook 02 — Two-Atom Rydberg Blockade

This notebook extends the single-atom model to **two interacting atoms** and studies the **Rydberg blockade** regime.

We use the Hamiltonian

$$
H = \sum_{i=1}^2 \left( \frac{\Omega}{2}\sigma_x^{(i)} - \Delta n_i \right) + V n_1 n_2
$$

where:
- $\Omega$ is the Rabi frequency,
- $\Delta$ is the detuning,
- $V$ is the Rydberg interaction strength,
- $n_i = |r\rangle\langle r|_i$ projects atom $i$ onto the excited / Rydberg-like state.

The main goal is to show how increasing $V$ suppresses double excitation, which is the hallmark of blockade.


In [ ]:
# Ensure dependencies are installed before imports
import importlib, subprocess, sys

def ensure(pkg):
    try:
        importlib.import_module(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg])

for pkg in ['qutip', 'numpy', 'scipy', 'matplotlib']:
    ensure(pkg)


## Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from qutip import basis, qeye, tensor, sigmax, mesolve

## Basis states and operators

In [ ]:
# Single-atom basis
g = basis(2, 0)
r = basis(2, 1)
I = qeye(2)
sx = sigmax()
n = r * r.dag()

# Two-atom basis states
gg = tensor(g, g)
gr = tensor(g, r)
rg = tensor(r, g)
rr = tensor(r, r)

# Operators on each site
sx1 = tensor(sx, I)
sx2 = tensor(I, sx)
n1 = tensor(n, I)
n2 = tensor(I, n)

# Double-excitation projector
n_rr = rr * rr.dag()

## Model definition

In [ ]:
def two_atom_hamiltonian(omega: float, delta: float, V: float):
    """Two-atom driven Hamiltonian with Rydberg interaction."""
    drive = 0.5 * omega * (sx1 + sx2)
    detuning = -delta * (n1 + n2)
    interaction = V * n1 * n2
    return drive + detuning + interaction


def simulate_blockade(omega: float, delta: float, V: float, t_final: float = 10.0, n_steps: int = 400):
    """Return time grid and expectation values for single and double excitation."""
    H = two_atom_hamiltonian(omega, delta, V)
    times = np.linspace(0.0, t_final, n_steps)
    result = mesolve(H, gg, times, [], [n1 + n2, n_rr])
    single_plus_double = np.real(result.expect[0])
    double_exc = np.real(result.expect[1])
    return times, single_plus_double, double_exc


def max_double_excitation(omega: float, delta: float, V: float, t_final: float = 10.0, n_steps: int = 400):
    """Maximum double-excitation probability during evolution."""
    _, _, p_rr = simulate_blockade(omega, delta, V, t_final=t_final, n_steps=n_steps)
    return float(np.max(p_rr))


## Time traces for different interaction strengths

In [ ]:
omega = 1.0
delta = 0.0
V_values = [0.0, 2.0, 6.0]

plt.figure(figsize=(8, 5))
for V in V_values:
    times, _, p_rr = simulate_blockade(omega, delta, V, t_final=12.0, n_steps=500)
    plt.plot(times, p_rr, label=fr"V={V}")

plt.xlabel("Time")
plt.ylabel(r"Double-excitation probability $P_{rr}$")
plt.title(r"Suppression of double excitation as interaction strength $V$ increases")
plt.legend()
plt.tight_layout()
plt.show()

## Blockade scan versus interaction strength

We now quantify blockade by plotting the **maximum** double-excitation probability during the evolution as a function of $V$.

In [ ]:
V_scan = np.linspace(0.0, 10.0, 80)
max_p_rr = [max_double_excitation(omega=1.0, delta=0.0, V=V, t_final=12.0, n_steps=400) for V in V_scan]

In [ ]:
plt.figure(figsize=(7, 4.5))
plt.plot(V_scan, max_p_rr)
plt.xlabel(r"Interaction strength $V$")
plt.ylabel(r"Max double-excitation probability $\max(P_{rr})$")
plt.title(r"Rydberg blockade strengthens as $V$ increases")
plt.tight_layout()
plt.show()

## 2D blockade landscape over $(\Omega, V)$

Next we scan over drive strength and interaction strength. This produces a compact picture of when blockade is weak or strong.

In [ ]:
omega_vals = np.linspace(0.2, 4.0, 55)
V_vals = np.linspace(0.0, 10.0, 55)

blockade_landscape = np.zeros((len(V_vals), len(omega_vals)))

for i, V in enumerate(V_vals):
    for j, omega in enumerate(omega_vals):
        blockade_landscape[i, j] = max_double_excitation(omega=omega, delta=0.0, V=V, t_final=10.0, n_steps=250)

In [ ]:
plt.figure(figsize=(7.5, 5.5))
im = plt.imshow(
    blockade_landscape,
    origin='lower',
    aspect='auto',
    extent=[omega_vals[0], omega_vals[-1], V_vals[0], V_vals[-1]],
)
plt.xlabel(r"Rabi frequency $\Omega$")
plt.ylabel(r"Interaction strength $V$")
plt.title(r"Maximum double excitation over $(\Omega, V)$")
plt.colorbar(im, label=r"$\max(P_{rr})$")
plt.tight_layout()
plt.show()

## Interpretation

This notebook shows the basic blockade trend:

- For **small** $V$, both atoms can be excited, so $P_{rr}$ can become large.
- For **large** $V$, the doubly excited state is shifted out of resonance.
- As a result, double excitation is suppressed, which is the defining signature of **Rydberg blockade**.

The $(\Omega, V)$ landscape is a useful first diagnostic for identifying operating regimes relevant to neutral-atom control and gate design.


## Optional next steps

Natural extensions include:

- adding detuning $\Delta$ as a third scan parameter,
- including spontaneous emission or dephasing with Lindblad operators,
- computing Bell-state or CZ-like gate metrics,
- converting interaction strength $V$ into a distance-dependent model such as $V \propto 1/R^6$.
